# Domain-Specific Tool Agents: SQL, CSV, and Code Interpreter

---

## Learning Objectives

By the end of this notebook you will understand how to:

1. **Design tools** for specific data domains (databases, spreadsheets, code execution).
2. **Build an SQL agent** that inspects a database schema before querying it.
3. **Build a CSV analyzer agent** that translates natural language into pandas operations.
4. **Build a code interpreter agent** that executes arbitrary Python in a sandboxed subprocess.
5. Appreciate the **common ReAct pattern** that underpins all three agents.

---

## Architecture Overview

All three agents in this notebook follow the same **ReAct** loop (Reason, Act, Observe):

```
User Question
     |
     v
[ LLM Reasoning ]  -- decides which tool to call
     |
     v
[ Tool Execution ]  -- runs the tool and captures output
     |
     v
[ LLM Reasoning ]  -- interprets the result, may call another tool or answer
     |
     v
Final Answer
```

What differs between agents is **which tools** they have access to. The agent loop itself is identical. We implement it here using the **Azure OpenAI Responses API** directly, without LangGraph or MAF, so you can see the raw mechanics.

## Section 1: Setup

We load credentials from a `.env` file and initialize the Azure OpenAI client. Every agent in this notebook shares the same client and the same agent loop structure.

In [1]:
import os
import json
import sqlite3
import subprocess
import sys
import tempfile
import io

import pandas as pd
from dotenv import load_dotenv
from openai import AzureOpenAI

load_dotenv()

# ── Azure OpenAI client ────────────────────────────────────────────────────────
client = AzureOpenAI(
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version="2024-12-01-preview",
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
)

MODEL = os.getenv("AZURE_OPENAI_MODEL", "gpt-4.1-mini")

print(f"Azure OpenAI client ready  |  Model: {MODEL}")

Azure OpenAI client ready  |  Model: gpt-4.1-mini


## Section 2: SQL Agent

### Schema-First Design

A common mistake when building SQL agents is to let the LLM guess the table and column names. This leads to hallucinated column names and broken queries. The correct approach is **schema-first**: the agent always inspects the database schema before writing SQL.

We define two tools:

| Tool | Purpose |
|------|---------|
| `get_schema` | Returns all table names and their columns from the SQLite database |
| `query_sqlite` | Executes a SQL query and returns results (with basic safety checks) |

We use the Chinook sample database, a music store dataset with tables for artists, albums, tracks, invoices, and more.

In [2]:
# ── Database path ─────────────────────────────────────────────────────────────
DB_PATH = os.path.join("..", "..", "utility", "chinook.db")
DB_PATH = os.path.abspath(DB_PATH)
print(f"Database: {DB_PATH}")
print(f"Exists:   {os.path.exists(DB_PATH)}")


# ── Tool 1: Get Schema ────────────────────────────────────────────────────────
def get_schema() -> str:
    """Return all table names and their column definitions from the database."""
    try:
        conn = sqlite3.connect(DB_PATH)
        cursor = conn.cursor()
        cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
        tables = cursor.fetchall()

        schema = {}
        for (table_name,) in tables:
            cursor.execute(f"PRAGMA table_info({table_name});")
            columns = cursor.fetchall()
            schema[table_name] = [
                {"name": col[1], "type": col[2]} for col in columns
            ]
        conn.close()
        return json.dumps(schema, indent=2)
    except Exception as e:
        return f"Schema Error: {e}"


# ── Tool 2: Query SQLite ──────────────────────────────────────────────────────
def query_sqlite(sql: str) -> str:
    """Execute a read-only SQL query and return the results."""
    try:
        # Basic guard against destructive statements
        dangerous = ["drop", "delete", "update", "insert", "alter"]
        if any(keyword in sql.lower() for keyword in dangerous):
            return "Blocked: only SELECT queries are allowed."

        conn = sqlite3.connect(DB_PATH)
        cursor = conn.cursor()
        cursor.execute(sql)
        rows = cursor.fetchall()
        columns = [desc[0] for desc in cursor.description] if cursor.description else []
        conn.close()

        if not rows:
            return "Query returned no results."
        # Format as a readable table
        header = " | ".join(columns)
        lines = [header, "-" * len(header)]
        for row in rows[:50]:  # Limit output to 50 rows
            lines.append(" | ".join(str(v) for v in row))
        if len(rows) > 50:
            lines.append(f"... ({len(rows)} total rows, showing first 50)")
        return "\n".join(lines)
    except Exception as e:
        return f"SQL Error: {e}"


# Quick test: inspect the schema
schema_output = get_schema()
print("Schema preview (first 500 chars):")
print(schema_output[:500])

Database: /Users/josephazar/Desktop/WORK/GITHUB/agentic-ai-langchain-maf-workshop/utility/chinook.db
Exists:   True
Schema preview (first 500 chars):
{
  "Album": [
    {
      "name": "AlbumId",
      "type": "INTEGER"
    },
    {
      "name": "Title",
      "type": "NVARCHAR(160)"
    },
    {
      "name": "ArtistId",
      "type": "INTEGER"
    }
  ],
  "Artist": [
    {
      "name": "ArtistId",
      "type": "INTEGER"
    },
    {
      "name": "Name",
      "type": "NVARCHAR(120)"
    }
  ],
  "Customer": [
    {
      "name": "CustomerId",
      "type": "INTEGER"
    },
    {
      "name": "FirstName",
      "type": "NVARCHAR(40)"
 


In [3]:
# ── SQL Agent: Tool definitions for the API ──────────────────────────────────
sql_tools = [
    {
        "type": "function",
        "function": {
            "name": "get_schema",
            "description": "Get the database schema: all table names and their columns with types.",
            "parameters": {"type": "object", "properties": {}, "required": []},
        },
    },
    {
        "type": "function",
        "function": {
            "name": "query_sqlite",
            "description": "Execute a read-only SQL SELECT query on the SQLite database and return results.",
            "parameters": {
                "type": "object",
                "properties": {
                    "sql": {
                        "type": "string",
                        "description": "The SQL SELECT query to execute.",
                    }
                },
                "required": ["sql"],
            },
        },
    },
]

# Map tool names to Python functions
sql_tool_map = {
    "get_schema": lambda **kwargs: get_schema(),
    "query_sqlite": lambda **kwargs: query_sqlite(kwargs["sql"]),
}


# ── Generic agent loop ────────────────────────────────────────────────────────
def run_agent(system_prompt: str, user_message: str, tools: list, tool_map: dict, max_turns: int = 10):
    """Run a ReAct-style agent loop using the Azure OpenAI Responses API.

    1. Send the conversation to the model.
    2. If the model returns tool_calls, execute them and feed results back.
    3. Repeat until the model returns a final text response (no tool_calls).
    """
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_message},
    ]

    for turn in range(max_turns):
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools,
        )
        assistant_msg = response.choices[0].message
        messages.append(assistant_msg)

        # If no tool calls, we have the final answer
        if not assistant_msg.tool_calls:
            print(f"\n[Agent completed in {turn + 1} turn(s)]")
            return assistant_msg.content

        # Execute each tool call
        for tc in assistant_msg.tool_calls:
            fn_name = tc.function.name
            fn_args = json.loads(tc.function.arguments)
            print(f"  -> Tool call: {fn_name}({fn_args})")

            result = tool_map[fn_name](**fn_args)
            messages.append(
                {"role": "tool", "tool_call_id": tc.id, "content": str(result)}
            )

    return "Agent reached maximum turns without a final answer."


print("Agent loop and SQL tools defined.")

Agent loop and SQL tools defined.


In [4]:
# ── Test the SQL Agent ───────────────────────────────────────────────────────
sql_system_prompt = (
    "You are a SQLite database assistant.\n\n"
    "RULES:\n"
    "- ALWAYS call get_schema first to understand the database structure.\n"
    "- Convert the user question into a SQL SELECT query.\n"
    "- Call query_sqlite to execute it.\n"
    "- Return a clear, concise answer based on the results.\n"
)

answer = run_agent(
    system_prompt=sql_system_prompt,
    user_message="How many tracks are in the database?",
    tools=sql_tools,
    tool_map=sql_tool_map,
)

print(f"\nAnswer: {answer}")

  -> Tool call: get_schema({})


  -> Tool call: query_sqlite({'sql': 'SELECT COUNT(*) AS TrackCount FROM Track;'})



[Agent completed in 3 turn(s)]

Answer: There are 3503 tracks in the database.


> **Framework implementations:** See `langgraph/sql_react_agent.py` for the full LangGraph implementation with `StateGraph` and `ToolNode`, and `maf/sql_react_agent.py` for the MAF version using `OpenAIChatClient().as_agent()`.

## Section 3: CSV Analyzer Agent

### The Four-Tool Pattern

CSV analysis requires a different tool set than SQL. The agent needs to:

1. **Load** the file into memory (`load_csv`)
2. **Inspect** its shape, columns, and types (`get_csv_info`)
3. **Query** it with pandas expressions (`query_csv`)

The `query_csv` tool uses `eval()` with a restricted scope that only exposes the DataFrame as `df` and the `pd` module. This prevents the model from importing arbitrary modules or accessing the filesystem.

### Safety Considerations Around eval()

Using `eval()` is inherently risky. In a production system you should:
- Run the code in a sandboxed subprocess (like the code interpreter in Section 4)
- Use an allow-list of permitted operations
- Set resource limits (memory, CPU time)

For this workshop, the restricted scope (`{"df": df, "pd": pd}`) is a reasonable trade-off between usability and safety.

In [5]:
# ── Global DataFrame store ────────────────────────────────────────────────────
_dataframes: dict[str, pd.DataFrame] = {}


# ── Tool: Load CSV ────────────────────────────────────────────────────────────
def load_csv(path: str) -> str:
    """Load a CSV file into memory for analysis."""
    try:
        df = pd.read_csv(path)
        _dataframes["current"] = df
        return (
            f"CSV loaded: {df.shape[0]} rows x {df.shape[1]} columns.\n"
            f"Columns: {', '.join(df.columns.tolist())}\n"
            f"Preview:\n{df.head(3).to_string(index=False)}"
        )
    except Exception as e:
        return f"Error loading CSV: {e}"


# ── Tool: Get CSV Info ────────────────────────────────────────────────────────
def get_csv_info() -> str:
    """Return metadata: shape, column names, dtypes, and basic statistics."""
    if "current" not in _dataframes:
        return "No CSV loaded yet. Call load_csv first."
    df = _dataframes["current"]
    buf = io.StringIO()
    df.info(buf=buf)
    return (
        f"Shape: {df.shape}\n\n"
        f"Dtypes:\n{df.dtypes.to_string()}\n\n"
        f"First 5 rows:\n{df.head().to_string()}\n\n"
        f"Numeric summary:\n{df.describe().to_string()}"
    )


# ── Tool: Query CSV ──────────────────────────────────────────────────────────
def query_csv(code: str) -> str:
    """Execute a pandas expression on the loaded DataFrame.

    The DataFrame is available as `df`. The `pd` module is also available.
    Example: df.groupby('product')['quantity'].sum().sort_values(ascending=False)
    """
    if "current" not in _dataframes:
        return "No CSV loaded yet. Call load_csv first."
    df = _dataframes["current"]
    try:
        # Restricted scope: only df and pd are accessible
        result = eval(code, {"__builtins__": {}}, {"df": df, "pd": pd})
        if isinstance(result, (pd.DataFrame, pd.Series)):
            return result.to_string()
        return str(result)
    except Exception as e:
        return f"Error: {e}"


print("CSV tools defined: load_csv, get_csv_info, query_csv")

CSV tools defined: load_csv, get_csv_info, query_csv


In [6]:
# ── Create a sample CSV for testing ──────────────────────────────────────────
sample_csv = """date,product,quantity,price
2024-01-15,Widget A,120,9.99
2024-01-15,Widget B,80,14.99
2024-01-15,Widget C,45,24.99
2024-02-15,Widget A,150,9.99
2024-02-15,Widget B,95,14.99
2024-02-15,Widget C,60,24.99
2024-03-15,Widget A,130,9.99
2024-03-15,Widget B,110,14.99
2024-03-15,Widget C,75,24.99
2024-04-15,Widget A,140,9.99
2024-04-15,Widget B,100,14.99
2024-04-15,Widget C,55,24.99"""

csv_path = os.path.join(tempfile.gettempdir(), "sales_data.csv")
with open(csv_path, "w") as f:
    f.write(sample_csv)

print(f"Sample CSV written to: {csv_path}")
print(pd.read_csv(csv_path).to_string(index=False))

Sample CSV written to: /var/folders/71/7mh3_2nx4rg1xs9mhzb6zq0r0000gn/T/sales_data.csv
      date  product  quantity  price
2024-01-15 Widget A       120   9.99
2024-01-15 Widget B        80  14.99
2024-01-15 Widget C        45  24.99
2024-02-15 Widget A       150   9.99
2024-02-15 Widget B        95  14.99
2024-02-15 Widget C        60  24.99
2024-03-15 Widget A       130   9.99
2024-03-15 Widget B       110  14.99
2024-03-15 Widget C        75  24.99
2024-04-15 Widget A       140   9.99
2024-04-15 Widget B       100  14.99
2024-04-15 Widget C        55  24.99


In [7]:
# ── CSV Agent: Tool definitions for the API ─────────────────────────────────
csv_tools = [
    {
        "type": "function",
        "function": {
            "name": "load_csv",
            "description": "Load a CSV file from the given path into memory for analysis.",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {"type": "string", "description": "File path to the CSV."}
                },
                "required": ["path"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_csv_info",
            "description": "Get metadata about the loaded CSV: shape, columns, dtypes, and summary statistics.",
            "parameters": {"type": "object", "properties": {}, "required": []},
        },
    },
    {
        "type": "function",
        "function": {
            "name": "query_csv",
            "description": (
                "Execute a pandas expression on the loaded DataFrame. "
                "The DataFrame is available as `df`. The `pd` module is available. "
                "Example: df.groupby('product')['quantity'].sum()"
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "code": {
                        "type": "string",
                        "description": "A pandas expression using `df` and optionally `pd`.",
                    }
                },
                "required": ["code"],
            },
        },
    },
]

csv_tool_map = {
    "load_csv": lambda **kwargs: load_csv(kwargs["path"]),
    "get_csv_info": lambda **kwargs: get_csv_info(),
    "query_csv": lambda **kwargs: query_csv(kwargs["code"]),
}

# ── Test the CSV Agent ────────────────────────────────────────────────────────
csv_system_prompt = (
    "You are a CSV data analyst assistant.\n\n"
    "RULES:\n"
    "- First load the CSV file using load_csv.\n"
    "- Use get_csv_info to understand the data structure.\n"
    "- Translate questions into pandas expressions and use query_csv.\n"
    "- Return clear, concise answers with the actual numbers.\n"
    f"\nThe CSV file is located at: {csv_path}\n"
)

answer = run_agent(
    system_prompt=csv_system_prompt,
    user_message="What product had the highest total revenue?",
    tools=csv_tools,
    tool_map=csv_tool_map,
)

print(f"\nAnswer: {answer}")

  -> Tool call: load_csv({'path': '/var/folders/71/7mh3_2nx4rg1xs9mhzb6zq0r0000gn/T/sales_data.csv'})


  -> Tool call: query_csv({'code': "df['revenue'] = df['quantity'] * df['price']\nhighest_revenue_product = df.groupby('product')['revenue'].sum().idxmax()\nhighest_revenue_value = df.groupby('product')['revenue'].sum().max()\n{'product': highest_revenue_product, 'revenue': highest_revenue_value}"})


  -> Tool call: query_csv({'code': "df['revenue'] = df['quantity'] * df['price']\nrevenue_by_product = df.groupby('product')['revenue'].sum()\nhighest_revenue_product = revenue_by_product.idxmax()\nhighest_revenue_value = revenue_by_product.max()\nresult = {'product': highest_revenue_product, 'revenue': highest_revenue_value}\nresult"})


  -> Tool call: query_csv({'code': "df['revenue'] = df['quantity'] * df['price']\nrevenue_by_product = df.groupby('product')['revenue'].sum()\nrevenue_by_product"})


  -> Tool call: query_csv({'code': "df['revenue'] = df['quantity'] * df['price']\ndf.groupby('product')['revenue'].sum()"})


...

  -> Tool call: get_csv_info({})


  -> Tool call: query_csv({'code': "df['revenue'] = df['quantity'] * df['price']\nrevenue_by_product = df.groupby('product')['revenue'].sum()\nrevenue_by_product.idxmax()"})


  -> Tool call: query_csv({'code': "df['revenue'] = df['quantity'] * df['price']\ndf[['product','revenue']]"})


  -> Tool call: query_csv({'code': "df.groupby('product').apply(lambda x: (x['quantity'] * x['price']).sum())"})

Answer: Agent reached maximum turns without a final answer.


<string>:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.


> **Framework implementations:** See `langgraph/CSV-File-Analyzer.py` and `maf/CSV-File-Analyzer.py` for the full LangGraph and MAF versions with additional visualization tools.

## Section 4: Code Interpreter Agent

### Sandboxing Strategy

The code interpreter lets the LLM write and execute arbitrary Python code. This is powerful but dangerous. Our sandboxing approach:

1. **Write to a temp file** -- the code never runs in the notebook's own process.
2. **Execute via `subprocess.run()`** -- isolated process with no access to our variables.
3. **Enforce a timeout** -- kills runaway code after 15 seconds.
4. **Inject matplotlib backend** -- `matplotlib.use('Agg')` is prepended so plots save to files instead of trying to open GUI windows.
5. **Clean up** -- the temp file is deleted after execution.

This is not a production-grade sandbox (for that, use Docker containers or a service like E2B), but it is sufficient for workshop exercises.

In [8]:
# ── Tool: Execute Python ─────────────────────────────────────────────────────
def execute_python(code: str) -> str:
    """Write Python code to a temp file and execute it in a subprocess.

    Returns stdout on success, or stderr on failure.
    Timeout: 15 seconds.
    """
    with tempfile.NamedTemporaryFile(
        mode="w", suffix=".py", delete=False, encoding="utf-8"
    ) as f:
        # Inject non-interactive matplotlib backend so savefig() works
        preamble = "import matplotlib\nmatplotlib.use('Agg')\n"
        f.write(preamble + code)
        tmp_path = f.name

    try:
        result = subprocess.run(
            [sys.executable, tmp_path],
            capture_output=True,
            text=True,
            timeout=15,
        )
        stdout = result.stdout.strip()
        stderr = result.stderr.strip()

        if result.returncode != 0:
            return f"Error:\n{stderr}"

        return stdout if stdout else "Code executed successfully (no output)."

    except subprocess.TimeoutExpired:
        return "Execution timed out (>15 seconds). Simplify the code."
    finally:
        os.unlink(tmp_path)


# ── Code Interpreter: Tool definition for the API ────────────────────────────
code_tools = [
    {
        "type": "function",
        "function": {
            "name": "execute_python",
            "description": (
                "Write and execute Python code. Use this for calculations, data analysis, "
                "transformations, or generating plots. For plots, use matplotlib and call "
                "plt.savefig('output.png') instead of plt.show()."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "code": {
                        "type": "string",
                        "description": "The Python code to execute.",
                    }
                },
                "required": ["code"],
            },
        },
    }
]

code_tool_map = {
    "execute_python": lambda **kwargs: execute_python(kwargs["code"]),
}

print("Code interpreter tool defined.")

Code interpreter tool defined.


In [9]:
# ── Test the Code Interpreter Agent ──────────────────────────────────────────
code_system_prompt = (
    "You are a Python code interpreter agent.\n\n"
    "RULES:\n"
    "- ALWAYS write and execute Python code to solve problems.\n"
    "- Never compute answers manually -- use the execute_python tool.\n"
    "- For plots, use matplotlib and call plt.savefig('output.png').\n"
    "- If code fails, read the error, fix it, and try again.\n"
    "- After execution, explain the result clearly.\n"
)

answer = run_agent(
    system_prompt=code_system_prompt,
    user_message="Calculate the first 20 Fibonacci numbers and print them.",
    tools=code_tools,
    tool_map=code_tool_map,
)

print(f"\nAnswer: {answer}")

  -> Tool call: execute_python({'code': 'def fibonacci(n):\n    fib_seq = [0, 1]\n    while len(fib_seq) < n:\n        fib_seq.append(fib_seq[-1] + fib_seq[-2])\n    return fib_seq\n\nfirst_20_fib = fibonacci(20)\nfirst_20_fib'})



[Agent completed in 2 turn(s)]

Answer: The first 20 Fibonacci numbers have been calculated. Here they are:

[0, 1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89, 144, 233, 377, 610, 987, 1597, 2584, 4181]


### Discussion: Security and Limitations

**Timeout:** The 15-second timeout prevents infinite loops and expensive computations. In production, you might implement escalating timeouts (e.g., 5s for simple math, 30s for data processing, 120s for ML training).

**Security:** Our subprocess sandbox is basic. The code can still access the filesystem and network. For real applications, consider:
- Running inside a Docker container with no network access
- Using a dedicated service like E2B or Modal
- Restricting imports with a custom import hook

**Matplotlib backend injection:** We prepend `matplotlib.use('Agg')` to every script so that `plt.show()` does not hang waiting for a GUI window. The agent is instructed to use `plt.savefig()` instead.

> **Framework implementations:** See `langgraph/code-interpreter.py` and `maf/code-interpreter.py` for the full implementations.

## Section 5: LangGraph vs MAF Comparison

Both frameworks in this workshop implement the same ReAct pattern, but they differ in how they express it. Here is a side-by-side comparison:

| Aspect | LangGraph | MAF |
|--------|-----------|-----|
| **Tool definition** | `@tool` decorator with docstring | Plain Python function |
| **Agent loop** | `StateGraph` with explicit nodes and edges | `OpenAIChatClient().as_agent()` |
| **State management** | `TypedDict` with `add_messages` reducer | Implicit message history |
| **Error handling** | Custom error-tracker node or conditional edges | Decorator-based retry |
| **Tool execution** | `ToolNode` (prebuilt) or custom node | Automatic dispatch by the framework |
| **Memory** | `MemorySaver` checkpointer | Built-in conversation context |

### When to use which?

- **LangGraph** gives you fine-grained control over the execution graph. Use it when you need custom routing, parallel tool execution, or complex multi-agent workflows.
- **MAF** minimizes boilerplate. Use it when you want to prototype quickly or when the standard ReAct loop is sufficient.

The raw OpenAI API approach shown in this notebook sits between the two: maximum transparency, but more code to maintain.

## Section 6: Summary and Exercises

### Key Insights

1. **Schema-first for SQL** -- Always let the agent inspect the database structure before writing queries. This eliminates hallucinated table and column names.
2. **Restricted eval for CSV** -- Expose only `df` and `pd` in the eval scope. This balances flexibility (the model can write any pandas expression) with safety (no filesystem or network access).
3. **Subprocess sandboxing for code** -- Run untrusted code in a separate process with a timeout. Inject the matplotlib backend to prevent GUI hangs.
4. **The agent loop is universal** -- The `run_agent` function we built works with any tool set. The only things that change between agents are the tools and the system prompt.

---

### Exercises

**(a) Add a `list_tables` tool to the SQL agent**

Create a lightweight tool that returns only the table names (not the full schema). Update the system prompt to instruct the agent to call `list_tables` first, then `get_schema` only for the relevant tables. This reduces token usage on large databases.

**(b) Add a `save_results` tool to the CSV agent**

Implement a tool that takes a pandas expression and an output path, executes the expression, and saves the resulting DataFrame to a new CSV file. Test it by asking the agent to filter the sales data and save the results.

**(c) Add timeout escalation to the code interpreter**

Modify `execute_python` to accept an optional `timeout` parameter (default 15 seconds). Update the tool definition so the model can request more time for complex computations. Add a hard cap of 60 seconds that cannot be overridden.